In [6]:
#Installments
!pip install pandas
!pip install numpy
!pip install matplotlib
!pip install seaborn
!pip install scikit-learn
!pip install imbalanced-learn
!pip install category-encoders


[notice] A new release of pip is available: 25.0.1 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 25.0.1 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 25.0.1 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 25.0.1 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 25.0.1 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 25.0.1 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 25.0.1 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


# Section 1

In [7]:
import pandas as pd

# Lista de CSVs de entrenamiento
df_files = [
    "archive/CIC_IoT_Part_1.csv",
    "archive/CIC_IoT_Part_2.csv",
    "archive/Lab_1.csv",
    "archive/Lab_2.csv",
    "archive/UNSW_IoT_Traces.csv"
]

# Cargar cada dataset por separado en una lista de DataFrames
dfs = []
for f in df_files:
    df_temp = pd.read_csv(f, low_memory=False)
    dfs.append(df_temp)

print(f"Cargados {len(dfs)} datasets:")
for i, df in enumerate(dfs):
    print(f"Dataset {i+1}: {df.shape}")

Cargados 5 datasets:
Dataset 1: (1000000, 89)
Dataset 2: (707918, 89)
Dataset 3: (38125, 89)
Dataset 4: (88692, 89)
Dataset 5: (933833, 89)


In [8]:
# Eliminar columnas que no se necesitan o causan data leakage
cols_to_drop = ['Unnamed: 0', 'FlowID', 'Source', 'SrcIP', 'DstIP', 
                'Timestamp', 'MAC', 'connection_type', 'DeviceName']

# Aplicar a todos los datasets
for i, df in enumerate(dfs):
    # Eliminar solo las columnas que existan en el dataframe
    cols_existentes = [col for col in cols_to_drop if col in df.columns]
    dfs[i] = df.drop(columns=cols_existentes)
    print(f"Dataset {i+1}: {dfs[i].shape} (eliminadas {len(cols_existentes)} columnas)")

print(f"\n✅ Columnas eliminadas de todos los datasets")

Dataset 1: (1000000, 80) (eliminadas 9 columnas)
Dataset 2: (707918, 80) (eliminadas 9 columnas)
Dataset 3: (38125, 80) (eliminadas 9 columnas)
Dataset 4: (88692, 80) (eliminadas 9 columnas)
Dataset 5: (933833, 80) (eliminadas 9 columnas)

✅ Columnas eliminadas de todos los datasets


In [9]:
print(dfs[0].columns.tolist())

['SrcPort', 'DstPort', 'Protocol', 'FlowDuration', 'TotFwdPkts', 'TotBwdPkts', 'TotLenFwdPkts', 'TotLenBwdPkts', 'FwdPktLenMax', 'FwdPktLenMin', 'FwdPktLenMean', 'FwdPktLenStd', 'BwdPktLenMax', 'BwdPktLenMin', 'BwdPktLenMean', 'BwdPktLenStd', 'FlowByts/s', 'FlowPkts/s', 'FlowIATMean', 'FlowIATStd', 'FlowIATMax', 'FlowIATMin', 'FwdIATTot', 'FwdIATMean', 'FwdIATStd', 'FwdIATMax', 'FwdIATMin', 'BwdIATTot', 'BwdIATMean', 'BwdIATStd', 'BwdIATMax', 'BwdIATMin', 'FwdPSHFlags', 'BwdPSHFlags', 'FwdURGFlags', 'BwdURGFlags', 'FwdHeaderLen', 'BwdHeaderLen', 'FwdPkts/s', 'BwdPkts/s', 'PktLenMin', 'PktLenMax', 'PktLenMean', 'PktLenStd', 'PktLenVar', 'FINFlagCnt', 'SYNFlagCnt', 'RSTFlagCnt', 'PSHFlagCnt', 'ACKFlagCnt', 'URGFlagCnt', 'CWEFlagCount', 'ECEFlagCnt', 'Down/UpRatio', 'PktSizeAvg', 'FwdSegSizeAvg', 'BwdSegSizeAvg', 'FwdByts/bAvg', 'FwdPkts/bAvg', 'FwdBlkRateAvg', 'BwdByts/bAvg', 'BwdPkts/bAvg', 'BwdBlkRateAvg', 'SubflowFwdPkts', 'SubflowFwdByts', 'SubflowBwdPkts', 'SubflowBwdByts', 'InitFwd

In [ ]:
from sklearn.preprocessing import LabelEncoder

# 2. Definir Mapeo a Macro-Categorías (4 categorías basadas en patrones de tráfico)


class_to_macrocategory = {
    # 1. MULTIMEDIA (Alto ancho de banda, streaming, video)
    'Camera': 'MULTIMEDIA',
    'VideoDoorbell': 'MULTIMEDIA',
    'baby_monitor': 'MULTIMEDIA',
    'Audio': 'MULTIMEDIA',
    'TV': 'MULTIMEDIA',
    
    # 2. SMART_CONTROLS (Control, automatización, actuadores)
    'Lighting': 'SMART_CONTROLS',
    'Plug': 'SMART_CONTROLS',
    'PowerOutlet': 'SMART_CONTROLS',
    'power_switch': 'SMART_CONTROLS',
    'Hub': 'SMART_CONTROLS',
    'Fan': 'SMART_CONTROLS',
    'Vacuum_Cleaner': 'SMART_CONTROLS',
    'coffee_maker': 'SMART_CONTROLS',
    'Motion_Sensor': 'SMART_CONTROLS',
    
    # 3. SENSORS (Bajo tráfico, datos periódicos)
    'Weather': 'SENSORS',
    'AirPurifier': 'SENSORS',
    'Humidifer': 'SENSORS',
    'Scale': 'SENSORS',
    'sleep_sensor': 'SENSORS',
    'SmartBoard': 'SENSORS',
    
    # 4. COMPUTING (Dispositivos de propósito general)
    'router': 'COMPUTING',
    'smartphone': 'COMPUTING',
    'printer': 'COMPUTING',
    'PC': 'COMPUTING',
}

print("\n=== Aplicando Macro-Categorías ===")
for i, df in enumerate(dfs):
    # Usamos replace para mapear. Los valores que no estén en el dict se quedan igual.
    # Si 'Type' contiene el tipo de dispositivo, se agrupará.
    df['Type'] = df['Type'].replace(class_to_macrocategory)
    dfs[i] = df

# 3. Label Encoding (Estandarizar la variable 'Type' a números)
print("\n=== Label Encoding (Type) ===")
all_types = set()
for df in dfs:
    all_types.update(df['Type'].unique())

sorted_classes = sorted(list(all_types))
le = LabelEncoder()
le.fit(sorted_classes)
print(f"Clases finales ({len(le.classes_)}): {le.classes_}")

for i, df in enumerate(dfs):
    df['Type'] = le.transform(df['Type'])
    dfs[i] = df
    
print("✅ Datos limpios, categorías agrupadas y etiquetas codificadas.")



=== Aplicando Macro-Categorías ===

=== Label Encoding (Type) ===
Clases finales (4): ['COMPUTING' 'MULTIMEDIA' 'SENSORS' 'SMART_CONTROLS']
✅ Datos limpios, categorías agrupadas y etiquetas codificadas.


In [11]:
# Analisis de distribución de Macro-Categorías por dataset
print("\n=== Distribución de Macro-Categorías por Dataset ===")

for i, df in enumerate(dfs):
    filename = df_files[i]
    print(f"\nDataset {i+1}: {filename}")
    
    # Obtener conteo de clases (que ahora son números gracias al LabelEncoder)
    counts = df['Type'].value_counts().sort_index()
    
    # Recuperar los nombres reales usando el encoder (le)
    if 'le' in locals():
        class_indices = counts.index
        class_names = le.inverse_transform(class_indices)
        
        # Mostrar tabla
        print(f"{'Macro-Categoría':<25} | {'Muestras':<10} | {'% del Dataset':<10}")
        print("-" * 55)
        for name, count in zip(class_names, counts):
            percent = (count / len(df)) * 100
            print(f"{name:<25} | {count:<10} | {percent:.2f}%")
            
    else:
        print(counts)


=== Distribución de Macro-Categorías por Dataset ===

Dataset 1: archive/CIC_IoT_Part_1.csv
Macro-Categoría           | Muestras   | % del Dataset
-------------------------------------------------------
MULTIMEDIA                | 759500     | 75.95%
SENSORS                   | 12301      | 1.23%
SMART_CONTROLS            | 228199     | 22.82%

Dataset 2: archive/CIC_IoT_Part_2.csv
Macro-Categoría           | Muestras   | % del Dataset
-------------------------------------------------------
COMPUTING                 | 73003      | 10.31%
MULTIMEDIA                | 471902     | 66.66%
SENSORS                   | 5649       | 0.80%
SMART_CONTROLS            | 157364     | 22.23%

Dataset 3: archive/Lab_1.csv
Macro-Categoría           | Muestras   | % del Dataset
-------------------------------------------------------
COMPUTING                 | 7663       | 20.10%
MULTIMEDIA                | 29782      | 78.12%
SMART_CONTROLS            | 680        | 1.78%

Dataset 4: archive/Lab_2.cs

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score
import numpy as np

# 5-Fold Cross-Validation Manual (cada dataset es un fold)
print("=== 5-Fold Cross-Validation con Random Forest ===\n")

accuracies = []
f1_weighted_scores = []
f1_macro_scores = []

for test_idx in range(len(dfs)):
    print(f"--- FOLD {test_idx + 1}/5 ---")
    print(f"Test: Dataset {test_idx + 1} ({df_files[test_idx]})")
    
    # Seleccionar datasets de entrenamiento (todos menos el de test)
    train_indices = [i for i in range(len(dfs)) if i != test_idx]
    print(f"Train: Datasets {[i+1 for i in train_indices]}")
    
    # Concatenar datasets de entrenamiento
    train_dfs = [dfs[i] for i in train_indices]
    train_data = pd.concat(train_dfs, ignore_index=True)
    
    # Dataset de test
    test_data = dfs[test_idx].copy()
    
    # Separar features y target
    X_train = train_data.drop(columns=['Type'])
    y_train = train_data['Type']
    X_test = test_data.drop(columns=['Type'])
    y_test = test_data['Type']
    
    print(f"Train shape: {X_train.shape}, Test shape: {X_test.shape}")
    
    # Entrenar Random Forest
    rf = RandomForestClassifier(n_estimators=50, random_state=42, n_jobs=-1, max_depth=15)
    rf.fit(X_train, y_train)
    
    # Predecir
    y_pred = rf.predict(X_test)
    
    # Calcular métricas
    acc = accuracy_score(y_test, y_pred)
    f1_weighted = f1_score(y_test, y_pred, average='weighted')
    f1_macro = f1_score(y_test, y_pred, average='macro')
    
    accuracies.append(acc)
    f1_weighted_scores.append(f1_weighted)
    f1_macro_scores.append(f1_macro)
    
    print(f"Accuracy: {acc:.4f}")
    print(f"F1-Score (weighted): {f1_weighted:.4f}")
    print(f"F1-Score (macro): {f1_macro:.4f}\n")

# Resultados finales
print("=" * 50)
print("=== RESULTADOS FINALES ===")
print(f"Accuracy promedio:        {np.mean(accuracies):.4f} ± {np.std(accuracies):.4f}")
print(f"F1-Score (weighted):      {np.mean(f1_weighted_scores):.4f} ± {np.std(f1_weighted_scores):.4f}")
print(f"F1-Score (macro):         {np.mean(f1_macro_scores):.4f} ± {np.std(f1_macro_scores):.4f}")
print("\nDesglose por fold:")
for i, (acc, f1w, f1m) in enumerate(zip(accuracies, f1_weighted_scores, f1_macro_scores)):
    print(f"Fold {i+1}: Accuracy={acc:.4f}, F1-Weighted={f1w:.4f}, F1-Macro={f1m:.4f}")


=== 5-Fold Cross-Validation con Random Forest ===

--- FOLD 1/5 ---
Test: Dataset 1 (archive/CIC_IoT_Part_1.csv)
Train: Datasets [2, 3, 4, 5]
Train shape: (1768568, 79), Test shape: (1000000, 79)
Accuracy: 0.9175
F1-Score (weighted): 0.9158
F1-Score (macro): 0.4544

--- FOLD 2/5 ---
Test: Dataset 2 (archive/CIC_IoT_Part_2.csv)
Train: Datasets [1, 3, 4, 5]
Train shape: (2060650, 79), Test shape: (707918, 79)
Accuracy: 0.7826
F1-Score (weighted): 0.7722
F1-Score (macro): 0.5341

--- FOLD 3/5 ---
Test: Dataset 3 (archive/Lab_1.csv)
Train: Datasets [1, 2, 4, 5]
Train shape: (2730443, 79), Test shape: (38125, 79)
Accuracy: 0.7292
F1-Score (weighted): 0.7353
F1-Score (macro): 0.6347

--- FOLD 4/5 ---
Test: Dataset 4 (archive/Lab_2.csv)
Train: Datasets [1, 2, 3, 5]
Train shape: (2679876, 79), Test shape: (88692, 79)


In [ ]:
# Entrenar modelo final con todos los datos y guardarlo
import joblib
from sklearn.ensemble import RandomForestClassifier

print("="*70)
print("=== ENTRENAMIENTO DEL MODELO FINAL CON TODOS LOS DATOS ===")
print("="*70)

# Concatenar todos los datasets
all_data = pd.concat(dfs, ignore_index=True)
print(f"\n📊 Dataset completo: {all_data.shape}")
print(f"   Muestras totales: {len(all_data):,}")
print(f"   Features: {all_data.shape[1] - 1}")
print(f"   Clases: {len(all_data['Type'].unique())}")

# Separar features y target
X = all_data.drop(columns=['Type'])
y = all_data['Type']

print(f"\n🔧 Entrenando Random Forest...")
print(f"   Parámetros: n_estimators=50, max_depth=15, n_jobs=-1")

# Entrenar el modelo final
final_model = RandomForestClassifier(
    n_estimators=50, 
    random_state=42, 
    n_jobs=-1, 
    max_depth=15
)

final_model.fit(X, y)

# Guardar el modelo y el encoder
model_filename = 'iot_device_classifier_rf.pkl'
encoder_filename = 'label_encoder.pkl'

joblib.dump(final_model, model_filename)
joblib.dump(le, encoder_filename)

print(f"\n✅ Modelo entrenado exitosamente!")
print(f"   Archivo del modelo: {model_filename}")
print(f"   Archivo del encoder: {encoder_filename}")

# Información adicional del modelo
print(f"\n📈 Información del modelo:")
print(f"   Número de árboles: {final_model.n_estimators}")
print(f"   Profundidad máxima: {final_model.max_depth}")
print(f"   Features usados: {final_model.n_features_in_}")
print(f"   Clases: {le.classes_}")

# Accuracy en el conjunto completo (como referencia)
y_pred_train = final_model.predict(X)
train_acc = accuracy_score(y, y_pred_train)
print(f"\n   Accuracy en datos de entrenamiento: {train_acc:.4f}")

print("\n" + "="*70)
print("Para cargar el modelo más tarde, usa:")
print("  model = joblib.load('iot_device_classifier_rf.pkl')")
print("  encoder = joblib.load('label_encoder.pkl')")
print("="*70)
